In [24]:
from backtesting import Backtest, Strategy
import pandas as pd
import math

In [25]:
df = pd.read_csv("xauusd-m1-bid-2020-01-01-2026-08-15.csv")

df.set_index("Datetime", inplace=True)

df = df[["Open", "High", "Low", "Close"]]

df.index = pd.to_datetime(df.index, unit="ms")

df = df.tail(50000)

df

,Open,High,Low,Close
Datetime,,,,
2026-07-03 05:34:00,4178.745,4179.355,4177.635,4178.475
2026-07-03 05:35:00,4178.495,4179.425,4177.535,4177.685
2026-07-03 05:36:00,4177.665,4178.135,4175.935,4176.905
2026-07-03 05:37:00,4176.885,4178.245,4176.455,4177.815
2026-07-03 05:38:00,4177.715,4177.735,4174.025,4174.025
...,...,...,...,...
2026-08-14 20:55:00,4375.265,4375.335,4374.715,4375.065
2026-08-14 20:56:00,4375.065,4375.065,4374.685,4374.915
2026-08-14 20:57:00,4374.975,4375.215,4374.585,4375.085


In [26]:
def get_digits(price_step):
    return round(-math.log10(price_step))

In [27]:
BALANCE         = 100_000_000     # starting balance

CONTRACT_SIZE   = 100
LOT_SIZE        = 0.01

GAP             = 5
N_LEVELS        = 5  
    
MAX_LOSS        = 10     
PROFIT_TARGET   = 10

# e.g. EURUSD=X has a price step of 0.0001
# e.g. XAUUSD has a price step of 0.01
TICKER_PRICE_PRECISION = get_digits(0.01)

In [28]:
class GridTrendStrategy(Strategy):

    # -----------------------------
    # Grid parameters
    # -----------------------------
    n_levels = N_LEVELS
    grid_gap = GAP
    units_per_order = LOT_SIZE * CONTRACT_SIZE

    # -----------------------------
    # Risk parameters ($)
    # -----------------------------
    profit_target = PROFIT_TARGET
    max_loss = MAX_LOSS

    def init(self):
        self.grid_center = None

        self.buy_orders = []
        self.sell_orders = []

        self.reset_grid(self.data.Close[-1])

    def reset_grid(self, center_price):
        # Cancel pending orders from previous grid
        for order in list(self.orders):
            order.cancel()

        self.buy_orders = []
        self.sell_orders = []

        self.grid_center = center_price

        # -----------------------------
        # stop orders grid
        # -----------------------------
        for i in range(1, self.n_levels + 1):
            self.buy_orders.append(self.buy(
                size=self.units_per_order,
                stop=center_price + i * self.grid_gap,
            ))
            self.sell_orders.append(self.sell(
                size=self.units_per_order,
                stop=center_price - i * self.grid_gap,
            ))

    def basket_pnl(self):
        # Total floating PnL of all open trades.
        return sum(trade.pl for trade in self.trades)

    def close_basket(self):
        # Close all open positions and rebuild the grid around current price.
        for trade in list(self.trades): trade.close()

        self.reset_grid(self.data.Close[-1])

    def next(self):
        pnl = self.basket_pnl()

        # -----------------------------
        # Profit target: +$5
        # -----------------------------
        if pnl >= self.profit_target:
            self.close_basket()

        # -----------------------------
        # Maximum loss: -$10
        # -----------------------------
        if pnl <= -self.max_loss:
            self.close_basket()

        return

# ============================================================
# Backtest
# ============================================================

bt = Backtest(
    df,
    GridTrendStrategy,
    cash=BALANCE,
    margin=1/1000,
    commission=0.0,
    exclusive_orders=False,
    hedging=True,
    finalize_trades=True
)

In [29]:
stats = bt.run()

print(stats)

Start                     2026-07-03 05:34:00
End                       2026-08-14 20:59:00
Duration                     42 days 15:25:00
Exposure Time [%]                      99.998
Equity Final [$]                100006962.218
Equity Peak [$]                 100009551.108
Return [%]                            0.00696
Buy & Hold Return [%]                 4.68424
Return (Ann.) [%]                      0.0726
Volatility (Ann.) [%]                 0.02287
CAGR [%]                              0.05962
Sharpe Ratio                          3.17455
Sortino Ratio                         8.22633
Calmar Ratio                         18.20938
Alpha [%]                             0.00645
Beta                                  0.00011
Max. Drawdown [%]                    -0.00399
Avg. Drawdown [%]                    -0.00028
Max. Drawdown Duration       22 days 13:42:00
Avg. Drawdown Duration        0 days 09:13:00
# Trades                                  823
Win Rate [%]                      

In [30]:
bt.plot()

/home/mod7ex/projects/algo-strategies/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '10min'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1829', ...)